# Unsloth Qwen3.5-9B LoRA Trainer [Checkpoint-Resume]

This notebook trains a tiny LoRA adapter on your newly generated synthetic computer-use dataset using the highly-optimized Unsloth library. It natively supports Unsloth's extreme MoE bf16 optimizations to fit on standard GPUs.

## Step 0 — Imports

Core libraries. `FastLanguageModel` is Unsloth's main entry-point for loading, patching, and saving models. `safetensors` is used later to load adapter weights directly from disk.

In [ ]:
import os
import json
import torch
from unsloth import FastLanguageModel  # Unsloth-patched model loader / saver

## Step 1 — Configuration

**Edit these paths before running.** Everything else in the notebook is derived from them.

| Variable | Description |
|---|---|
| `BASE_MODEL_PATH` | Local folder containing the full Qwen3.5-9B base weights (`config.json` + `model.safetensors-*`) |
| `ADAPTER_PATH` | Unsloth `checkpoint-900` directory — adapter-only, no base weights |
| `MERGED_OUTPUT` | Where the merged fp16 model will be written |
| `GGUF_OUTPUT` | Where the final quantised GGUF file will land |

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Priority: Env var (OI_MCP_PATH) > Current directory discovery
def get_base_path():
    path = os.environ.get("OI_MCP_PATH")
    if path:
        return path.strip('"\'')
    # Default to current repo root if no env var is found
    # We assume we're running from the scripts/ folder inside computer-use-finetuning
    return os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

BASE_DIR = get_base_path()
MODELS_DIR = os.path.join(BASE_DIR, "computer-use-finetuning", "models")

# Full Qwen3.5-9B base weights (config.json + model.safetensors-* shards)
BASE_MODEL_PATH = os.path.join(MODELS_DIR, "qwen3.5-9b-base-model")

# Unsloth adapter-only checkpoint from training run (adapter_config.json + adapter_model.safetensors)
ADAPTER_PATH    = os.path.join(MODELS_DIR, "adapter-checkpoint")

# Output: standard HF model with LoRA baked in (fp16, ~18 GB)
MERGED_OUTPUT   = os.path.join(MODELS_DIR, "base-lora-merged")

# Output: quantised GGUF for llama.cpp / Ollama (~5.5 GB for Q4_K_M)
GGUF_OUTPUT     = os.path.join(MODELS_DIR, "qwen3.5-9b-tool-use-lora-gguf")

# ── Model hyperparameters ──────────────────────────────────────────────────────
MAX_SEQ_LENGTH  = 4096   # Must match what was used during training

print(f"Base dir   : {BASE_DIR}")
print(f"Base model : {BASE_MODEL_PATH}")
print(f"Adapter    : {ADAPTER_PATH}")
print(f"Merged out : {MERGED_OUTPUT}")
print(f"GGUF out   : {GGUF_OUTPUT}")

## Step 2 — Load Base Model

Load the full Qwen3.5-9B base weights via Unsloth's `FastLanguageModel`.

- `load_in_4bit=False` is mandatory here — quantised weights cannot be merged cleanly into a fp16 model.
- This step reads the 4× safetensors shards from `BASE_MODEL_PATH` (∼19 GB) and applies Unsloth's kernel patches.

In [ ]:
# Load the base Qwen3.5-9B model using Unsloth's optimised loader.
# Unsloth will apply its custom CUDA kernels (FlashAttention, fused RoPE, etc.)
# automatically. No adapter is attached at this stage.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_PATH,  # local model-backup folder
    max_seq_length = MAX_SEQ_LENGTH,   # same context length used during training
    dtype          = torch.bfloat16,   # bfloat16 throughout for clean fp16 merge
    load_in_4bit   = False,            # MUST be False — 4bit quantisation prevents merging
)
print("Base model loaded successfully.")

## Step 3 — Attach LoRA Adapter Structure

Wrap the base model with the same LoRA configuration that was used during training.

> **Why this step is needed:** Unsloth's `get_peft_model` builds its own internal LoRA graph with optimised CUDA kernels — it does **not** use vanilla PEFT's graph. We must re-create this graph here so that when we load the adapter weights in the next step, they slot into exactly the same parameter names that Unsloth wrote during training.

> All values below (`r`, `target_modules`, `lora_alpha`, etc.) **must exactly match** the values used in the original training run.

In [ ]:
# Re-create the exact same LoRA graph that was used during training.
# Unsloth builds its own optimised LoRA parameter names internally;
# these names must match what was written to adapter_model.safetensors.
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,                 # LoRA rank
    target_modules             = [                   # Attention + FFN projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha                 = 16,                 # Scaling factor (= r → effective scale = 1.0)
    lora_dropout               = 0,                  # 0 is heavily optimised by Unsloth
    bias                       = "none",             # No bias terms in adapter
    use_gradient_checkpointing = "unsloth",          # Unsloth's memory-efficient checkpointing
    random_state               = 3407,               # Reproducibility seed
)
print("LoRA adapter structure attached.")

## Step 4 — Load Trained Adapter Weights

Read the trained LoRA weight tensors from `adapter_model.safetensors` and inject them into the model graph created in the previous step.

- `strict=False` is intentional — safetensors files written by Unsloth's trainer contain only the LoRA delta matrices; all other keys (base weights, norms, embeddings) are already in the model from Step 2 and are correctly ignored.
- After this step the model in memory is effectively **base weights + fine-tuned LoRA deltas**.

In [ ]:
from safetensors.torch import load_file
import torch

# ── Step 4: Load Trained Adapter Weights with Key Remapping ─────────────────────
#
# Root cause of original failure:
#   Adapter file keys  : ...lora_A.weight  (no adapter name segment)
#   Model graph keys   : ...lora_A.default.weight  (Unsloth inserts 'default')
#
# Fix: remap every key by inserting '.default' before the final '.weight'
#      i.e.  lora_A.weight  ->  lora_A.default.weight
#            lora_B.weight  ->  lora_B.default.weight

adapter_weights_path = os.path.join(ADAPTER_PATH, "adapter_model.safetensors")
file_sd = load_file(adapter_weights_path)

# Remap keys
remapped_sd = {}
for k, v in file_sd.items():
    # Insert 'default' adapter name before the trailing .weight
    # e.g.  base_model...lora_A.weight  ->  base_model...lora_A.default.weight
    new_k = k.replace("lora_A.weight", "lora_A.default.weight") \
              .replace("lora_B.weight", "lora_B.default.weight")
    remapped_sd[new_k] = v

# Inject into the model - only update LoRA params, keep everything else
result = model.load_state_dict(remapped_sd, strict=False)
print(f"Missing keys  : {len(result.missing_keys)}  (base weights - expected)")
print(f"Unexpected keys: {len(result.unexpected_keys)}  (should be 0 now)")

if result.unexpected_keys:
    print("  Still-unexpected sample:", result.unexpected_keys[:3])
else:
    print("All 256 LoRA delta tensors injected successfully!")

## Step 5 — Merge LoRA into Base and Save fp16

Fuse the LoRA delta matrices permanently into the base weight tensors and write the result as a standard HuggingFace model.

- `FastLanguageModel.for_inference(model)` disables training-only features (gradient checkpointing, dropout) and primes Unsloth's inference kernels.
- `save_method="merged_16bit"` walks every layer, computes `W_merged = W_base + (lora_B @ lora_A) * (alpha / r)`, and writes the result in bfloat16 safetensors shards.
- Output size: ~18 GB (same as original base but now includes fine-tuned behaviour).

In [ ]:
import os, json, shutil, glob
import torch
from safetensors.torch import load_file, save_file
from tqdm import tqdm

# ── Manual shard-by-shard LoRA merge ─────────────────────────────────────────
#
# Why: Unsloth's save_pretrained_merged reads lora_weights from the model's
# internal lora_layer objects, which are still meta tensors even after
# load_state_dict succeeds (PEFT registers them differently internally).
#
# This approach:
#   1. Reads LoRA deltas directly from the adapter safetensors file on disk.
#   2. Reads each base model shard directly from disk (never touches model obj).
#   3. Applies W_merged = W_base + lora_B @ lora_A * (alpha / r) on GPU.
#   4. Writes merged shards to MERGED_OUTPUT.
#   5. Copies over all non-weight files (config, tokenizer, etc.).

os.makedirs(MERGED_OUTPUT, exist_ok=True)

# ── 1. Load LoRA deltas from adapter file ────────────────────────────────────
print("Loading LoRA adapter weights...")
adapter_file = os.path.join(ADAPTER_PATH, "adapter_model.safetensors")
lora_sd = load_file(adapter_file, device="cpu")

# Parse adapter_config.json for r and lora_alpha
with open(os.path.join(ADAPTER_PATH, "adapter_config.json")) as f:
    adapter_cfg = json.load(f)
lora_r     = adapter_cfg["r"]
lora_alpha = adapter_cfg["lora_alpha"]
scale      = lora_alpha / lora_r
print(f"LoRA scale (alpha/r) = {lora_alpha}/{lora_r} = {scale}")

# Build lookup: base weight name -> (lora_A tensor, lora_B tensor)
# Adapter keys: base_model.model.X.lora_A.weight
# We want to map to: X.weight  (the key inside the base model safetensors)
lora_map = {}  # key = bare weight path (e.g. model.language_model.layers.0.mlp.down_proj.weight)
for k in lora_sd.keys():
    if ".lora_A." not in k:
        continue
    # e.g. base_model.model.model.language_model.layers.0.mlp.down_proj.lora_A.weight
    # strip leading 'base_model.model.' and trailing '.lora_A.weight'
    bare = k.replace("base_model.model.", "", 1)   # model.language_model...
    bare = bare.replace(".lora_A.weight", ".weight")  # -> model.language_model...down_proj.weight
    lora_A_key = k
    lora_B_key = k.replace(".lora_A.", ".lora_B.")
    if lora_B_key in lora_sd:
        lora_map[bare] = (lora_sd[lora_A_key].to(torch.float32),
                          lora_sd[lora_B_key].to(torch.float32))

print(f"LoRA modules to merge: {len(lora_map)}")

# ── 2. Find base model safetensor shards ─────────────────────────────────────
index_file = os.path.join(BASE_MODEL_PATH, "model.safetensors.index.json")
if os.path.exists(index_file):
    with open(index_file) as f:
        index = json.load(f)
    weight_map = index["weight_map"]  # {param_name: shard_filename}
    shard_files = sorted(set(weight_map.values()))
else:
    # Single-file model
    shard_files = ["model.safetensors"]
    weight_map = None

print(f"Base model shards: {shard_files}")

# ── 3. Process each shard ─────────────────────────────────────────────────────
merged_count = 0
for shard_fname in tqdm(shard_files, desc="Merging shards"):
    shard_path = os.path.join(BASE_MODEL_PATH, shard_fname)
    print(f"  Loading {shard_fname}...")
    shard_sd = load_file(shard_path, device="cuda")

    merged_shard = {}
    for param_name, W in shard_sd.items():
        if param_name in lora_map:
            lora_A, lora_B = lora_map[param_name]
            lora_A = lora_A.to("cuda", dtype=torch.float32)
            lora_B = lora_B.to("cuda", dtype=torch.float32)
            W_f32  = W.to(torch.float32)
            # lora_B: (out, r), lora_A: (r, in)  ->  delta: (out, in)
            delta  = (lora_B @ lora_A) * scale
            W_merged = (W_f32 + delta).to(torch.bfloat16)
            merged_shard[param_name] = W_merged.cpu()
            merged_count += 1
        else:
            merged_shard[param_name] = W.cpu()

    out_path = os.path.join(MERGED_OUTPUT, shard_fname)
    save_file(merged_shard, out_path)
    print(f"  Saved {shard_fname} ({merged_count} LoRA merges so far)")
    del shard_sd, merged_shard
    torch.cuda.empty_cache()

print(f"\nTotal LoRA modules merged: {merged_count}  (expected {len(lora_map)})")

# ── 4. Copy index + config files ─────────────────────────────────────────────
for fname in os.listdir(BASE_MODEL_PATH):
    if fname.endswith(".safetensors"):
        continue   # already written above
    src = os.path.join(BASE_MODEL_PATH, fname)
    dst = os.path.join(MERGED_OUTPUT, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

# Also copy tokenizer files from ADAPTER_PATH (may have updated special tokens)
for fname in os.listdir(ADAPTER_PATH):
    if fname.startswith("adapter"):
        continue   # skip adapter weights/config
    src = os.path.join(ADAPTER_PATH, fname)
    dst = os.path.join(MERGED_OUTPUT, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print(f"Merged fp16 model saved to: {MERGED_OUTPUT}")

## Step 6 — Export to GGUF (Q4\_K\_M)

Convert the merged fp16 model to GGUF format and quantise it to **Q4\_K\_M** — the recommended general-purpose quantisation for llama.cpp and Ollama.

| Format | Size | Quality | Use case |
|---|---|---|---|
| `q4_k_m` | ~5.5 GB | Good | Default — best size/quality trade-off |
| `q5_k_m` | ~6.7 GB | Better | More headroom if VRAM allows |
| `q8_0` | ~9.5 GB | Near-lossless | Benchmarking / evaluation |
| `f16` | ~18 GB | Lossless | Full precision GGUF |

> Unsloth calls `llama.cpp`'s `convert_hf_to_gguf.py` and `quantize` binaries internally, so llama.cpp must be compiled and accessible on the system.

In [ ]:
os.makedirs(GGUF_OUTPUT, exist_ok=True)

# Convert and quantise the merged fp16 model to GGUF.
# Unsloth shells out to llama.cpp's convert_hf_to_gguf.py + quantize internally.
# Output file: {GGUF_OUTPUT}/unsloth.Q4_K_M.gguf  (~5.5 GB)
# Load with: ollama create qwen-tool-use --file Modelfile
#            or: llama-cli -m unsloth.Q4_K_M.gguf

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MERGED_OUTPUT,   # local merged folder
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = torch.bfloat16,
    load_in_4bit    = False,
)

# (Optional) FastLanguageModel.for_inference(model)

model.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer,
    quantization_method = "q4_k_m",    # or "q8_0" / "f16"
)

print(f"GGUF model saved to: {GGUF_OUTPUT}")